# Steam Indie Game Major Genre & Tag Analysis

이 노트북은 사용자 투표 기반의 **태그(Tags)**와 스팀 공식 분류인 **장르(Genres)** 데이터를 병합하여 인디 게임 시장의 주요 트렌드를 분석합니다.

### 분석 포인트
1. **공식 장르 vs 유저 태그**: 공식 분류된 게임군 내에서 유저들이 실제로 어떤 특징에 투표했는지 확인
2. **장르별 태그 분포**: 주요 장르별 상위 태그들의 인기(투표수) 분포 비교
3. **커스텀 필터링**: 특정 장르 혹은 복합 장르(Action + RPG 등) 내의 세부 특징 분석

## 1. 라이브러리 로드 및 환경 설정

In [207]:
import pandas as pd
import numpy as np
import json
import ast
import plotly.express as px
import plotly.graph_objects as go
from collections import Counter
import os

# 결과 확인을 위한 설정
pd.set_option('display.max_columns', None)

## 2. 데이터 로드 및 전처리 (Data Merging)

In [208]:
# 1. 파일 경로 설정
tags_file = "../../../data/processed/steam_indie_tags.csv"
sample_file = "../../../data/processed/steam_stratified_sample.csv"
# 2. 데이터 로드
# 사용자 태그 데이터 (header가 없으므로 컬럼명 지정)
tags_df = pd.read_csv(tags_file, names=['appid', 'name', 'dev', 'pub', 'owners', 'pos', 'neg', 'price', 'tags', 'updated_at'])
# 층화 추출된 샘플 데이터 (stratum 컬럼 포함)
sample_df = pd.read_csv(sample_file)

In [209]:
# 3. 데이터 병합 (appid 기준)
# sample_df의 모든 정보와 tags_df의 tags 컬럼을 합칩니다.
df = pd.merge(sample_df, tags_df[['appid', 'tags']], on='appid', how='inner')

In [210]:
# 4. 데이터 파싱
def parse_tags_dict(x):
    if pd.isna(x) or x == '{}': return {}
    try: return json.loads(x.replace("''", "'"))
    except:
        try: return ast.literal_eval(x)
        except: return {}
def parse_official_genres(x):
    if pd.isna(x): return []
    try: return ast.literal_eval(x)
    except: return []
df['tags_dict'] = df['tags'].apply(parse_tags_dict)
df['official_genres'] = df['genres'].apply(parse_official_genres)
print(f"분석 대상 샘플 게임 수: {len(df)}")
df[['name_store', 'stratum', 'official_genres', 'tags_dict']].head()

분석 대상 샘플 게임 수: 74


,name_store,stratum,official_genres,tags_dict
0,Sun Haven,large_high,"[Adventure, Casual, Indie, RPG, Simulation]","{'RPG': 442, 'Magic': 324, 'Combat': 287, 'Min..."
1,(the) Gnorp Apologue,large_high,"[Casual, Indie, Simulation, Strategy]","{'2D': 117, 'Cute': 133, 'Idler': 193, 'Indie'..."
2,轮回修仙路,large_high,"[Adventure, Indie, RPG, Simulation]","{'3D': 359, 'RPG': 409, 'Indie': 332, 'Space':..."
3,MiSide,large_high,"[Adventure, Indie, RPG, Simulation]","{'2D': 761, '3D': 1392, 'RPG': 815, 'Cute': 21..."
4,Necesse,large_high,"[Action, Adventure, Indie, RPG]","{'2D': 188, 'RPG': 241, 'Co-op': 237, 'Indie':..."


In [211]:
# 1. 전체 태그 통합 집계 (투표수 합계 및 등장 게임 수)
all_tags_votes = Counter()
all_tags_appearance = Counter()

for tags in df['tags_dict']:
    for tag, vote in tags.items():
        all_tags_votes[tag] += vote
        all_tags_appearance[tag] += 1

# 2. 통계 데이터프레임 생성
full_tag_stats = pd.DataFrame({
    'tag': list(all_tags_votes.keys()),
    'total_votes': list(all_tags_votes.values()),
    'game_count': [all_tags_appearance[t] for t in all_tags_votes.keys()]
})

# 3. 인기 순위 정렬 (투표수 기준 상위 20개)
top_popular_tags = full_tag_stats.sort_values(by='total_votes', ascending=False).head(20)

# 4. 시각화: 투표수(인기) vs 등장 빈도(대중성) 비교
fig_popular = px.bar(
    top_popular_tags, 
    x='total_votes', 
    y='tag', 
    orientation='h',
    color='game_count', # 색상은 얼마나 많은 게임에 등장했는지를 나타냄
    title='스팀 인디 게임 인기 태그 TOP 20 (투표수 기준)',
    labels={'total_votes': '총 투표수 (인기)', 'tag': '태그명', 'game_count': '등장 게임 수 (대중성)'},
    color_continuous_scale='Viridis',
    hover_data=['game_count']
)

fig_popular.update_layout(yaxis={'categoryorder':'total ascending'}, height=800)
fig_popular.show()

# 5. 텍스트 리스트 출력
print("--- [참고] 스팀 최고 인기 태그 상위 20위 ---")
for i, row in top_popular_tags.head(10).iterrows():
    print(f"{row['tag'].ljust(15)} | 총 투표수: {int(row['total_votes']):>8,} | 등장 게임 수: {int(row['game_count']):>4}")


--- [참고] 스팀 최고 인기 태그 상위 20위 ---
Singleplayer    | 총 투표수:   16,198 | 등장 게임 수:   57
Simulation      | 총 투표수:   10,552 | 등장 게임 수:   25
Strategy        | 총 투표수:    8,881 | 등장 게임 수:   24
3D              | 총 투표수:    8,289 | 등장 게임 수:   23
Story Rich      | 총 투표수:    7,434 | 등장 게임 수:   17
2D              | 총 투표수:    7,140 | 등장 게임 수:   29
RPG             | 총 투표수:    7,084 | 등장 게임 수:   19
Survival        | 총 투표수:    6,578 | 등장 게임 수:   15
Pixel Graphics  | 총 투표수:    6,383 | 등장 게임 수:   21
Adventure       | 총 투표수:    6,263 | 등장 게임 수:   27


In [212]:
# 1. 게임 수(대중성) 기준으로 상위 20개 정렬
top_global_tags = full_tag_stats.sort_values(by='game_count', ascending=False).head(20)

# 2. 시각화 (스팀 공식 브라우저 스타일)
fig_global = px.bar(
    top_global_tags, 
    x='game_count', 
    y='tag', 
    orientation='h',
    color='total_votes', # 색상은 투표수(호응도)를 나타냄
    title='스팀 인디 게임 인기 태그 TOP 20 (게임 수 기준)',
    labels={'game_count': '해당 태그를 가진 게임 수', 'tag': '태그명', 'total_votes': '총 투표수'},
    color_continuous_scale='Viridis'
)

fig_global.update_layout(yaxis={'categoryorder':'total ascending'}, height=800)
fig_global.show()

# 3. 공식 페이지 순위와 비교 출력
print("--- [참고] 스팀 공식 기준 상위 20위 태그 ---")
for i, (idx, row) in enumerate(top_global_tags.head(10).iterrows(), 1):
    print(f"{i}위: {row['tag'].ljust(15)} (게임 수: {int(row['game_count']):>4})")


--- [참고] 스팀 공식 기준 상위 20위 태그 ---
1위: Singleplayer    (게임 수:   57)
2위: Indie           (게임 수:   38)
3위: 2D              (게임 수:   29)
4위: Action          (게임 수:   28)
5위: Adventure       (게임 수:   27)
6위: Casual          (게임 수:   25)
7위: Simulation      (게임 수:   25)
8위: Strategy        (게임 수:   24)
9위: 3D              (게임 수:   23)
10위: Exploration     (게임 수:   21)


In [213]:
# 1. 계층 대분류 정의
strata_levels = ['large_high', 'mid_high', 'small_high']
# 2. 분석 대상 주요 장르 정의
major_genres = ['Action', 'Adventure', 'Casual', 'Simulation', 'RPG', 'Strategy']
# 3. 제외할 범용 태그
exclude_list = []

print("=== 계층별/장르별 주요 분석 후보 태그 선정 결과 ===\n")

final_selection_report = []

for level in strata_levels:
    print(f"[{level.upper()} 계층 분석 결과]")
    
    level_df = df[df['stratum'] == level].copy()
    
    for genre in major_genres:
        genre_in_level_df = level_df[level_df['official_genres'].apply(lambda x: genre in x)]
        
        if genre_in_level_df.empty:
            continue
            
        group_tag_votes = Counter()
        for t_dict in genre_in_level_df['tags_dict']:
            for tag, vote in t_dict.items():
                if tag != genre and tag not in exclude_list:
                    group_tag_votes[tag] += vote
        
        top_selected = [tag for tag, _ in sorted(group_tag_votes.items(), key=lambda x: x[1], reverse=True)[:5]]
        
        if top_selected:
            result_str = f"  - {genre.ljust(10)}: {', '.join(top_selected)}"
            print(result_str)
            
            final_selection_report.append({
                'Stratum_Level': level,
                'Base_Genre': genre,
                'Candidate_Tags': top_selected
            })
    print("-" * 50)

df_selection_summary = pd.DataFrame(final_selection_report)


=== 계층별/장르별 주요 분석 후보 태그 선정 결과 ===

[LARGE_HIGH 계층 분석 결과]
  - Action    : Singleplayer, PvE, Survival, Resource Management, Simulation
  - Adventure : Singleplayer, Simulation, Survival, Horror, Zombies
  - Casual    : Simulation, Singleplayer, Strategy, Resource Management, Base-Building
  - Simulation: Singleplayer, Strategy, Resource Management, Tactical, Base-Building
  - RPG       : Singleplayer, Story Rich, Pixel Graphics, Psychological Horror, 2D
  - Strategy  : Simulation, Singleplayer, Tactical, Resource Management, Survival
--------------------------------------------------
[MID_HIGH 계층 분석 결과]
  - Action    : Controller, Singleplayer, Cyberpunk, Action-Adventure, Horror
  - Adventure : Singleplayer, Third Person, 3D, Action-Adventure, RPG
  - Casual    : Cute, 2D, Rogue-lite, Singleplayer, Choices Matter
  - Simulation: Singleplayer, Strategy, Farming Sim, Base-Building, RPG
  - RPG       : 2D, Singleplayer, Rogue-lite, Strategy, Indie
  - Strategy  : Singleplayer, RPG, Simula

In [214]:
# 1. 계층별 태그 빈도 계산 함수
def get_tag_freq_by_stratum(stratum_name):
    stratum_df = df[df['stratum'] == stratum_name]
    total_games = len(stratum_df)
    
    tag_counts = Counter()
    tag_votes = Counter()
    for tags in stratum_df['tags_dict']:
        for tag, vote in tags.items():
            tag_counts[tag] += 1
            tag_votes[tag] += vote
            
    stats = pd.DataFrame({
        'tag': list(tag_counts.keys()),
        f'freq_{stratum_name}': [c / total_games for c in tag_counts.values()],
        f'avg_votes_{stratum_name}': [tag_votes[t] / tag_counts[t] for t in tag_counts.keys()]
    })
    return stats

# 2. large_high vs small_high 데이터 추출 및 병합
large_stats = get_tag_freq_by_stratum('large_high')
small_stats = get_tag_freq_by_stratum('small_high')

comparison_df = pd.merge(large_stats, small_stats, on='tag', how='inner')

comparison_df = comparison_df.dropna(subset=['tag'])
comparison_df = comparison_df[comparison_df['tag'].str.strip() != ""]

# 3. 성공 기여도(Lift) 계산
comparison_df['success_lift'] = (comparison_df['freq_large_high'] / comparison_df['freq_small_high']).round(2)

exclude = ['Sexual Content']
final_comp = comparison_df[~comparison_df['tag'].isin(exclude)].sort_values('success_lift', ascending=False).head(20)

# 4. 시각화
fig_lift = px.bar(
    final_comp, x='success_lift', y='tag', orientation='h',
    title='large_high 계층에서 유독 강세인 "흥행 견인" 태그 (small_high 계층 대비 출현 비중)',
    labels={'success_lift': '성공 기여도 (Lift)', 'tag': '태그명'},
    color='success_lift', color_continuous_scale='Reds',
    hover_data=['freq_large_high', 'freq_small_high', 'avg_votes_large_high']
)

fig_lift.update_layout(
    yaxis={'categoryorder':'total ascending', 'automargin': True},
    height=700,
    margin=dict(l=150)
)

fig_lift.add_vline(x=1, line_dash="dash", line_color="black", annotation_text="Base (1.0)")
fig_lift.show()

print("==== small_high 계층 대비 출현 비중이 large_high 계층에서 강세인 태그들 ====")
display(final_comp[['tag', 'freq_large_high', 'avg_votes_large_high']])


==== small_high 계층 대비 출현 비중이 large_high 계층에서 강세인 태그들 ====


,tag,freq_large_high,avg_votes_large_high
4,Crafting,0.206897,348.166667
14,Character Customization,0.206897,336.500000
54,Procedural Generation,0.172414,327.000000
77,Gore,0.172414,219.600000
11,Multiplayer,0.344828,318.400000
13,Pixel Graphics,0.482759,363.571429
95,FPS,0.137931,402.750000
86,Zombies,0.137931,775.500000
41,Horror,0.241379,522.714286
70,Choices Matter,0.103448,252.000000


In [215]:
# 1. 계층별 태그 빈도 계산 함수 (median 추가)
def get_tag_stats_refined(stratum_name):
    stratum_df = df[df['stratum'] == stratum_name]
    total_games = len(stratum_df)
    
    tag_votes_list = {}
    for tags in stratum_df['tags_dict']:
        for tag, vote in tags.items():
            if tag not in tag_votes_list: tag_votes_list[tag] = []
            tag_votes_list[tag].append(vote)
            
    stats = pd.DataFrame({
        'tag': list(tag_votes_list.keys()),
        f'freq_{stratum_name}': [len(votes) / total_games for votes in tag_votes_list.values()],
        f'median_votes_{stratum_name}': [np.median(votes) for votes in tag_votes_list.values()]
    })
    return stats

# 2. 분석 실행 (large_high vs small_high)
large_stats = get_tag_stats_refined('large_high')
small_stats = get_tag_stats_refined('small_high')
comparison_df = pd.merge(large_stats, small_stats, on='tag', how='inner')
comparison_df['success_lift'] = (comparison_df['freq_large_high'] / comparison_df['freq_small_high']).round(2)

# 3. 시각화 (중앙값 기준)
final_comp_median = comparison_df[~comparison_df['tag'].isin(exclude)].sort_values('success_lift', ascending=False).head(20)

fig_median = px.scatter(
    final_comp_median, 
    x='success_lift', 
    y='median_votes_large_high',
    text='tag', 
    size='freq_large_high', 
    color='median_votes_large_high',
    color_continuous_scale='Reds',
    title='유저 투표 수 VS 흥행 기여도 (large_high vs small_high)',
    labels={
        'success_lift': '성공 기여도 (Lift)', 
        'median_votes_large_high': '유저 호응도 중앙값'
    },
    template='plotly_white'
)
fig_median.update_layout(height=800)
fig_median.show()

report_df = final_comp_median.sort_values('success_lift', ascending=False).copy()

print(f"{'='*85}")
print(f"{'순위':<4} {'태그명':<20} {'기여도(Lift)':<15} {'중앙값(Votes)':<15} {'출현비중(large_high)':<20}")
print(f"{'-'*85}")

for i, (_, row) in enumerate(report_df.iterrows(), 1):
    tag = row['tag']
    lift = row['success_lift']
    median_v = row['median_votes_large_high']
    ratio = row['freq_large_high'] * 100
    print(f"{i:<5} {tag:<22} {lift:<17.2f} {int(median_v):<17,} {ratio:<15.1f}%")

print(f"{'='*85}")
print(f"* 기여도(Lift): small_high 계층 대비 large_high 계층에서 발견될 확률")
print(f"* 중앙값(Votes): 흥행작들 사이에서의 일반적인 유저 투표수")


순위   태그명                  기여도(Lift)       중앙값(Votes)      출현비중(large_high)    
-------------------------------------------------------------------------------------
1     Crafting               4.14              351               20.7           %
2     Character Customization 4.14              348               20.7           %
3     Procedural Generation  3.45              298               17.2           %
4     Gore                   3.45              110               17.2           %
5     Multiplayer            3.45              286               34.5           %
6     Pixel Graphics         3.22              281               48.3           %
7     FPS                    2.76              423               13.8           %
8     Zombies                2.76              465               13.8           %
9     Horror                 2.41              464               24.1           %
10    Medieval               2.07              316               10.3           %
11    Bullet H

In [216]:
# large_high 계층 상위 태그들의 투표수 분포 확인 (중앙값 vs 평균 판단용)
top_tags = final_comp_median['tag'].tolist()

dist_rows = []
for tag in top_tags:
    votes = [v for tags in df[df['stratum'] == 'large_high']['tags_dict']
             for t, v in tags.items() if t == tag]
    if votes:
        dist_rows.append({
            'tag': tag,
            'mean': round(np.mean(votes), 1),
            'median': round(np.median(votes), 1),
            'skew': round(pd.Series(votes).skew(), 2),
            'max': max(votes),
            'n': len(votes)
        })

dist_df = pd.DataFrame(dist_rows).sort_values('skew', ascending=False)

fig_dist = go.Figure()
fig_dist.add_trace(go.Bar(name='평균', x=dist_df['tag'], y=dist_df['mean'], marker_color='tomato'))
fig_dist.add_trace(go.Bar(name='중앙값', x=dist_df['tag'], y=dist_df['median'], marker_color='steelblue'))
fig_dist.update_layout(
    barmode='group',
    title='태그별 투표수: 평균 vs 중앙값 비교 (large_high 계층)',
    xaxis_title='태그',
    yaxis_title='투표수',
    height=500,
    template='plotly_white',
    legend=dict(orientation='h', y=1.05)
)
fig_dist.show()

print("태그별 투표수 분포 요약 (skew > 1 이면 우편향 → 중앙값 사용이 적절)")
print(dist_df[['tag', 'n', 'mean', 'median', 'skew', 'max']].to_string(index=False))


태그별 투표수 분포 요약 (skew > 1 이면 우편향 → 중앙값 사용이 적절)
                    tag  n   mean  median  skew  max
         Pixel Graphics 14  363.6   281.5  2.97 1664
                   Gore  5  219.6   110.0  2.10  753
               Survival  8  532.5   274.5  2.04 1919
                 Horror  7  522.7   464.0  1.85 1685
  Procedural Generation  5  327.0   298.0  1.79  522
                Zombies  4  775.5   465.5  1.76 1924
              Difficult  3  838.7   307.0  1.73 1956
            Bullet Hell  3  678.0   384.0  1.73 1284
               Tactical  3 1049.0   399.0  1.72 2445
               Medieval  3  832.3   316.0  1.70 1991
             Historical  3  866.7   438.0  1.59 1987
            Multiplayer 10  318.4   286.0  0.98  858
                Sandbox  8  398.9   340.0  0.86  717
      Turn-Based Combat  3  136.0   128.0  0.45  218
Character Customization  6  336.5   348.5  0.18  569
               Crafting  6  348.2   351.0  0.07  530
                    FPS  4  402.8   423.0 -0.22  611
 

In [217]:
# 1. 분석할 계층 정의
strata_groups = ['large_high', 'mid_high', 'small_high']
top_n = 15

strata_top_tags = []

for s_name in strata_groups:
    s_df = df[df['stratum'] == s_name]
    total_s_games = len(s_df)
    
    s_counts = Counter()
    for tags in s_df['tags_dict']:
        for t in tags.keys():
            if t not in ['Indie', 'Singleplayer']:
                s_counts[t] += 1
                
    top_tags = s_counts.most_common(top_n)
    for tag, count in top_tags:
        strata_top_tags.append({
            'Stratum': s_name,
            'Tag': tag,
            'Count': count,
            'Ratio': round(count / total_s_games * 100, 1)
        })

df_strata_rank = pd.DataFrame(strata_top_tags)

fig_rank = px.bar(
    df_strata_rank, 
    x='Ratio', 
    y='Tag', 
    facet_col='Stratum',
    color='Stratum',
    orientation='h',
    title='계층별 최다 등장 태그 순위 (전체 게임 대비 비중 %)',
    labels={'Ratio': '등장 비중 (%)', 'Tag': '태그명'},
    height=600
)

fig_rank.update_yaxes(matches=None, categoryorder='total ascending')
fig_rank.show()

for s_name in strata_groups:
    print(f"\n[{s_name.upper()} 계층 Top {top_n} 태그]")
    temp = df_strata_rank[df_strata_rank['Stratum'] == s_name]
    print(temp[['Tag', 'Count', 'Ratio']].to_string(index=False))



[LARGE_HIGH 계층 Top 15 태그]
           Tag  Count  Ratio
Pixel Graphics     14   48.3
        Casual     13   44.8
    Simulation     11   37.9
            2D     11   37.9
      Strategy     11   37.9
        Action     11   37.9
   Multiplayer     10   34.5
     Adventure      9   31.0
  First-Person      9   31.0
   Exploration      8   27.6
    Rogue-lite      8   27.6
            3D      8   27.6
       Sandbox      8   27.6
      Survival      8   27.6
           RPG      7   24.1

[MID_HIGH 계층 Top 15 태그]
           Tag  Count  Ratio
          Cute     11   44.0
            2D     11   44.0
        Action     10   40.0
     Adventure      8   32.0
      Strategy      8   32.0
   Exploration      7   28.0
           RPG      7   28.0
    Story Rich      7   28.0
            3D      7   28.0
    Simulation      7   28.0
    Rogue-lite      7   28.0
         Funny      7   28.0
Choices Matter      6   24.0
    Rogue-like      6   24.0
        Puzzle      5   20.0

[SMALL_HIGH 계층 Top 

In [218]:
# large_high 계층 최다 등장 태그 TOP 20
strata_list = ['large_high']
colors = ['#EF553B']

for i, s_name in enumerate(strata_list):
    s_df = df[df['stratum'] == s_name]
    total_games = len(s_df)
    
    counts = Counter()
    for tags in s_df['tags_dict']:
        for t in tags.keys():
            counts[t] += 1
                
    df_temp = pd.DataFrame(counts.most_common(20), columns=['Tag', 'Count'])
    df_temp['Ratio'] = (df_temp['Count'] / total_games * 100).round(1)
    
    fig = px.bar(
        df_temp, 
        x='Ratio', 
        y='Tag', 
        orientation='h',
        text='Ratio',
        title=f'[{s_name.upper()} 계층] 최다 등장 태그 TOP 20 (전체 {total_games}개 게임 중 비중 %)',
        labels={'Ratio': '등장 비중 (%)', 'Tag': '태그명'},
        color_discrete_sequence=[colors[i]]
    )
    
    fig.update_layout(
        yaxis={'categoryorder':'total ascending'},
        height=600,
        margin=dict(l=150),
        xaxis_range=[0, 100]
    )
    fig.update_traces(texttemplate='%{text}%', textposition='outside')
    fig.show()


In [219]:
# 전체 태그별 적용된 게임 수 집계
tag_game_count = Counter()
for tags in df['tags_dict']:
    for tag in tags.keys():
        tag_game_count[tag] += 1

tag_game_df = pd.DataFrame(tag_game_count.items(), columns=['tag', 'game_count'])
tag_game_df = tag_game_df.sort_values('game_count', ascending=False).reset_index(drop=True)
tag_game_df.index += 1

print(f"전체 태그 종류 수: {len(tag_game_df)}")
print(f"전체 분석 게임 수: {len(df)}\n")

fig_tag_count = px.bar(
    tag_game_df.head(30),
    x='game_count',
    y='tag',
    orientation='h',
    title='태그별 적용 게임 수 TOP 30',
    labels={'game_count': '적용된 게임 수', 'tag': '태그명'},
    color='game_count',
    color_continuous_scale='Blues',
    text='game_count'
)
fig_tag_count.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    height=800,
    template='plotly_white'
)
fig_tag_count.update_traces(textposition='outside')
fig_tag_count.show()

print(tag_game_df.rename_axis('순위')[['tag', 'game_count']].rename(columns={'tag': '태그명', 'game_count': '게임 수'}).to_string())


전체 태그 종류 수: 276
전체 분석 게임 수: 74



                                   태그명  게임 수
순위                                          
1                         Singleplayer    57
2                                Indie    38
3                                   2D    29
4                               Action    28
5                            Adventure    27
6                               Casual    25
7                           Simulation    25
8                             Strategy    24
9                                   3D    23
10                         Exploration    21
11                      Pixel Graphics    21
12                                Cute    20
13                                 RPG    19
14                          Rogue-lite    18
15                          Story Rich    17
16                          Rogue-like    16
17                         Multiplayer    16
18                            Relaxing    16
19                        Early Access    15
20                        First-Person    15
21        

In [ ]:
from scipy import stats

# 1. positive_rate 계산
df['positive_rate'] = df['positive'] / df['total_reviews']

# 2. 태그 0/1 인코딩
all_tags = list(tag_game_count.keys())
tag_matrix = pd.DataFrame(
    [{tag: (tag in row) for tag in all_tags} for row in df['tags_dict']],
    index=df.index
).astype(int)

# 3. 최소 5개 게임 이상에 등장한 태그만 분석
valid_tags = [t for t in all_tags if tag_game_count[t] >= 5]

# 4. Spearman 상관계수 계산
# 주 지표: total_reviews (Steam 공식 집계)
# 보조 지표: owners_lower (교차 검증용)
results = []
for tag in valid_tags:
    x = tag_matrix[tag]

    corr_reviews, p_reviews = stats.spearmanr(x, df['total_reviews'])
    corr_owners, p_owners = stats.spearmanr(x, df['owners_lower'])
    corr_pos_rate, p_pos_rate = stats.spearmanr(x, df['positive_rate'])

    results.append({
        'tag': tag,
        'corr_reviews': round(corr_reviews, 3),
        'p_reviews': round(p_reviews, 3),
        'corr_owners': round(corr_owners, 3),
        'p_owners': round(p_owners, 3),
        'corr_pos_rate': round(corr_pos_rate, 3),
        'p_pos_rate': round(p_pos_rate, 3),
    })

corr_df = pd.DataFrame(results)

# 5. 유의미한 태그만 필터링 (주 지표 또는 만족도 기준 p < 0.05)
sig_df = corr_df[(corr_df['p_reviews'] < 0.05) | (corr_df['p_pos_rate'] < 0.05)].copy()
sig_df = sig_df.sort_values('corr_reviews', ascending=False)

print(f"분석 태그 수: {len(valid_tags)} / 유의미한 태그 수 (p<0.05): {len(sig_df)}\n")

# 6. 시각화: total_reviews vs positive_rate 상관계수 산점도
fig_corr = px.scatter(
    sig_df,
    x='corr_reviews',
    y='corr_pos_rate',
    text='tag',
    color='corr_reviews',
    color_continuous_scale='RdBu',
    range_color=[-1, 1],
    title='태그별 흥행 지표 Spearman 상관계수 (total_reviews vs positive_rate, p<0.05)',
    labels={'corr_reviews': '리뷰 수(total_reviews) 상관계수', 'corr_pos_rate': '긍정비율(positive_rate) 상관계수'},
    template='plotly_white'
)
fig_corr.update_traces(textposition='top center')
fig_corr.add_hline(y=0, line_dash='dash', line_color='gray', opacity=0.4)
fig_corr.add_vline(x=0, line_dash='dash', line_color='gray', opacity=0.4)
fig_corr.update_layout(height=600)
fig_corr.show()

# 7. 테이블 출력
print("=== 태그별 Spearman 상관계수 (유의미한 태그, total_reviews 기준 정렬) ===")
print(f"{'태그명':<25} {'reviews 상관':>13} {'p값':>8} {'owners 상관':>12} {'p값':>8} {'pos_rate 상관':>14} {'p값':>8}")
print("-" * 92)
for _, row in sig_df.iterrows():
    r_sig = '*' if row['p_reviews'] < 0.05 else ' '
    o_sig = '*' if row['p_owners'] < 0.05 else ' '
    p_sig = '*' if row['p_pos_rate'] < 0.05 else ' '
    print(f"{row['tag']:<25} {row['corr_reviews']:>12.3f}{r_sig} {row['p_reviews']:>8.3f} {row['corr_owners']:>11.3f}{o_sig} {row['p_owners']:>8.3f} {row['corr_pos_rate']:>13.3f}{p_sig} {row['p_pos_rate']:>8.3f}")
print("\n* p < 0.05 (통계적으로 유의미)")
print("* 주 지표: total_reviews (Steam 공식 집계) / 보조: owners_lower (교차 검증)")


In [ ]:
# Lift 분석 + 상관계수 분석 교집합 태그 추출
# - Lift > 1 (large_high 계층에서 더 자주 등장)
# - total_reviews 상관계수 > 0 이고 p < 0.05

lift_tags = set(final_comp_median['tag'].tolist())
corr_tags = set(sig_df[(sig_df['corr_reviews'] > 0) & (sig_df['p_reviews'] < 0.05)]['tag'].tolist())
common_tags = lift_tags & corr_tags

print(f"Lift 분석 태그 수: {len(lift_tags)}")
print(f"상관계수 유의미 태그 수 (reviews > 0, p<0.05): {len(corr_tags)}")
print(f"교집합 태그 수: {len(common_tags)}\n")

# 교집합 태그 상세 정보 병합
lift_info = final_comp_median[['tag', 'success_lift', 'freq_large_high', 'median_votes_large_high']]
corr_info = sig_df[['tag', 'corr_reviews', 'p_reviews', 'corr_owners', 'p_owners', 'corr_pos_rate', 'p_pos_rate']]

common_df = pd.merge(lift_info, corr_info, on='tag')
common_df = common_df[common_df['tag'].isin(common_tags)].sort_values('corr_reviews', ascending=False)

# 시각화
fig_common = px.scatter(
    common_df,
    x='success_lift',
    y='corr_reviews',
    text='tag',
    size='freq_large_high',
    color='corr_pos_rate',
    color_continuous_scale='RdBu',
    range_color=[-1, 1],
    title='흥행 기여도(Lift) & 상관계수 교집합 태그 (x: Lift, y: total_reviews 상관계수)',
    labels={
        'success_lift': '흥행 기여도 (Lift)',
        'corr_reviews': '리뷰 수 상관계수 (total_reviews)',
        'corr_pos_rate': '만족도 상관계수 (positive_rate)'
    },
    template='plotly_white'
)
fig_common.update_traces(textposition='top center')
fig_common.update_layout(height=600)
fig_common.show()

print("=== 흥행 기여도(Lift) & 상관계수 교집합 태그 ===")
print(f"{'태그명':<25} {'Lift':>8} {'reviews 상관':>13} {'p값':>8} {'owners 상관':>12} {'p값':>8} {'pos_rate 상관':>14} {'p값':>8}")
print("-" * 100)
for _, row in common_df.iterrows():
    r_sig = '*' if row['p_reviews'] < 0.05 else ' '
    o_sig = '*' if row['p_owners'] < 0.05 else ' '
    p_sig = '*' if row['p_pos_rate'] < 0.05 else ' '
    print(f"{row['tag']:<25} {row['success_lift']:>8.2f} {row['corr_reviews']:>12.3f}{r_sig} {row['p_reviews']:>8.3f} {row['corr_owners']:>11.3f}{o_sig} {row['p_owners']:>8.3f} {row['corr_pos_rate']:>13.3f}{p_sig} {row['p_pos_rate']:>8.3f}")

print("\n* p < 0.05 (통계적으로 유의미)")
print("* 주 지표: total_reviews / 보조: owners_lower (교차 검증)")
print("* 두 분석 모두에서 흥행과 관련성이 관찰된 태그")


In [222]:
# 게임 수 중앙값보다 적게 적용된 태그 출력
median_game_count = tag_game_df['game_count'].median()
below_median_df = tag_game_df[tag_game_df['game_count'] < median_game_count].copy()

print(f"전체 태그 수: {len(tag_game_df)}")
print(f"게임 수 중앙값: {int(median_game_count)}")
print(f"중앙값 미만 태그 수: {len(below_median_df)}\n")

print(below_median_df.rename_axis('순위')[['tag', 'game_count']].rename(columns={'tag': '태그명', 'game_count': '게임 수'}).to_string())


전체 태그 수: 276
게임 수 중앙값: 3
중앙값 미만 태그 수: 122

                         태그명  게임 수
순위                                
155       Twin Stick Shooter     2
156                  Cooking     2
157                    Short     2
158               Text-Based     2
159              Time Travel     2
160              Class-Based     2
161                 Abstract     2
162          Dungeon Crawler     2
163               3D Fighter     2
164                  Dragons     2
165             Lovecraftian     2
166                    Logic     2
167                   Rhythm     2
168          Time Management     2
169                   Demons     2
170                    Memes     2
171                  Fishing     2
172                      RTS     2
173                Narrative     2
174              Escape Room     2
175               Team-Based     2
176               Souls-like     2
177            Political Sim     2
178             Level Editor     2
179                Political     2
180         